# DEMO: Delta Table Statistics and Optimizations

In Databricks, **Delta Lake** provides data compression, segment elimination, and in-memory organization capabilities at the storage layer — and understanding them helps you write faster queries.

In this demo, you'll learn:
- How to inspect table physical details with `DESCRIBE DETAIL`
- How **data skipping** uses min/max statistics to avoid reading unnecessary files
- What `OPTIMIZE` does (file compaction) and why it matters
- How **Liquid Clustering** organizes data for your query patterns
- How **Predictive Optimization** handles all of this automatically

> **Note:** Run the setup cell below (Cell 2) to create the demo tables, then open the **SQL Editor** (click "SQL Editor" in the left sidebar) to run the remaining queries there.

---

In [0]:
%run ./Setup

## Part 1: DESCRIBE DETAIL — See the Physical Reality of Your Table

Every Delta table is actually a collection of **Parquet files** stored in cloud object storage. `DESCRIBE DETAIL` reveals what's happening at the physical level:

| Column | What it tells you |
| --- | --- |
| **numFiles** | How many data files make up this table |
| **sizeInBytes** | Total storage size of the table |
| **clusteringColumns** | Which columns the table is clustered by (if any) |
| **partitionColumns** | Which columns the table is partitioned by (legacy approach) |
| **properties** | Table settings including statistics configuration |

**Power BI Parallel:** This is like checking a Power BI dataset's storage mode and compression — it reveals the physical characteristics that affect query performance.

> Copy the queries below into the SQL Editor. Make sure you first run `USE SCHEMA <your_schema_name>;`

---

In [0]:
%sql
-- ============================================================
-- DESCRIBE DETAIL: Compare unclustered vs clustered tables
-- ============================================================
-- Run each statement separately to see the difference.
-- Pay attention to: numFiles, sizeInBytes, clusteringColumns

-- Table 1: Written with multiple appends, NO clustering
DESCRIBE DETAIL sales_events;

In [0]:
%sql

-- Table 2: Written once with CLUSTER BY (sale_date, region)
DESCRIBE DETAIL sales_events_clustered;

## Part 2: Data Skipping — The Most Important Optimization to Understand

Delta Lake automatically maintains **min/max statistics** for every column in every file. When you query with a WHERE clause, the engine checks these statistics *before* reading any data:

- If a file's max `sale_date` is `2024-03-31` and you filter for `sale_date >= '2024-10-01'`, that entire file is **skipped** without reading a single row.

**The more specific your WHERE clauses, the less data gets read.**

| Your Query | What Happens |
| --- | --- |
| `SELECT * FROM sales_events` (no filter) | Full scan — reads ALL files |
| `WHERE region = 'North'` | Skips files where max(region) < 'North' or min(region) > 'North' |
| `WHERE sale_date >= '2024-10-01'` | Skips files that can't contain October+ dates |

**Key insight:** By default, Delta Lake collects statistics on the **first 32 columns** defined in your table schema. This is controlled by the `delta.dataSkippingNumIndexedCols` table property.

---

In [0]:
%sql
-- ============================================================
-- Data Skipping in action: EXPLAIN with and without filters
-- ============================================================
-- Compare the DataFilters in the scan operator.
-- With a filter: Spark can skip files that don't match.
-- Without a filter: every file must be read (full scan).

-- Query WITH a filter (data skipping can help)
EXPLAIN
SELECT region, SUM(amount) AS total_revenue
FROM sales_events
WHERE sale_date >= '2024-10-01'
  AND region = 'North'
GROUP BY region;

In [0]:
%sql
-- ============================================================
-- Compare: Same query WITHOUT a filter (full table scan)
-- ============================================================
-- Notice: no DataFilters in the scan = reads ALL files.
-- This is the #1 performance mistake in dashboard queries.

EXPLAIN
SELECT region, SUM(amount) AS total_revenue
FROM sales_events
GROUP BY region;

   
## Bonus: Viewing the Statistics Delta Has Collected

The min/max statistics that power data skipping are stored in the **transaction log** (`_delta_log/`). Here's how to inspect them:

| Approach | What you'll see |
| --- | --- |
| `SHOW TBLPROPERTIES` | How many columns have statistics (`delta.dataSkippingNumIndexedCols`, default: 32) |
| `DESCRIBE HISTORY` | When `OPTIMIZE` / `ANALYZE` / `VACUUM` was last run |
| Read `_delta_log` (Python) | The exact min/max values stored per file, per column |

**Spotting tables that need OPTIMIZE:** Databricks targets ~128 MB per file. If the average file size is under 64 MB, `OPTIMIZE` will consolidate those files and improve data skipping effectiveness — more data per file means the min/max range is wider, but there are fewer files to open.

---

In [0]:
%sql
    
-- ============================================================
-- Check statistics configuration and maintenance history
-- ============================================================
-- Run each statement separately.

-- 1. Which columns have min/max statistics collected?
--    NOTE: If delta.dataSkippingNumIndexedCols is absent from the
--    output, the table is using the default of 32 (first 32 schema
--    columns). See the markdown cell below for both properties.
SHOW TBLPROPERTIES sales_events;

   
### Data Skipping: Properties That Control Statistics Collection

| Property | Behavior |
| --- | --- |
| `delta.dataSkippingNumIndexedCols` | Collects min/max stats on the **first N columns** in schema definition order. Default is **32**. Only appears in `SHOW TBLPROPERTIES` if explicitly set — absence means the default of 32 is active. Raise it if a filter column sits beyond position 32 in the schema. |
| `delta.dataSkippingStatsColumns` | Collects min/max stats on **exactly the columns you name**, regardless of schema position. **Overrides** `dataSkippingNumIndexedCols` when set. Preferred for wide tables or when filter columns are defined late in the schema. |

To set either property:
```sql
-- Position-based (raise the limit)
ALTER TABLE sales_events SET TBLPROPERTIES ('delta.dataSkippingNumIndexedCols' = '64');

-- Name-based (target specific columns — preferred)
ALTER TABLE sales_events SET TBLPROPERTIES ('delta.dataSkippingStatsColumns' = 'sale_date,region,amount');
```
> Changes only apply to files written **after** the property is set. Run `OPTIMIZE` to rewrite existing files with the updated statistics.

In [0]:
%sql
    
-- ============================================================
-- Target specific columns for statistics (preferred approach)
-- ============================================================
-- Instead of indexing "the first N columns", name exactly which
-- columns should have min/max statistics collected.
-- When set, this overrides dataSkippingNumIndexedCols entirely.

-- Set statistics collection to only your filter columns:
ALTER TABLE sales_events
SET TBLPROPERTIES ('delta.dataSkippingStatsColumns' = 'sale_date,region,amount');

-- Verify it was applied (the property will now appear in output):
SHOW TBLPROPERTIES sales_events;

-- IMPORTANT: This only affects files written AFTER the change.
-- To apply stats to existing files, run OPTIMIZE afterward:
--   OPTIMIZE sales_events;

In [0]:
%sql
    
-- 2. See what maintenance operations have been run on this table.
--    Look for: OPTIMIZE, VACUUM, ANALYZE, and WRITE operations.
--    If you see no OPTIMIZE entries, the table has never been compacted.
DESCRIBE HISTORY sales_events LIMIT 10;

In [0]:
# ============================================================
# File health check + peek at actual per-file statistics
# ============================================================
from pyspark.sql.functions import col, get_json_object

# --- Part A: Is this table a good OPTIMIZE candidate? ---
detail     = spark.sql("DESCRIBE DETAIL sales_events").first()
num_files  = detail["numFiles"]
size_bytes = detail["sizeInBytes"]
location   = detail["location"]

avg_mb = (size_bytes / max(num_files, 1)) / (1024**2)

print(f"Table     : sales_events")
print(f"Files     : {num_files:,}")
print(f"Total size: {size_bytes / (1024**2):.2f} MB")
print(f"Avg file  : {avg_mb:.2f} MB  (Databricks target: ~128 MB)")
print()
if avg_mb < 64:
    print("Small files detected — this table is a good OPTIMIZE candidate")
else:
    print("File sizes are healthy")

## Part 3: OPTIMIZE — File Compaction

As data is written to a table (especially via streaming or frequent appends), **many small files** accumulate. This hurts performance because:
- Each file has overhead to open and read
- Too many files means too much metadata to scan
- Data skipping becomes less effective (statistics per file are coarser)

The `OPTIMIZE` command **compacts small files into larger, more efficient ones**. Think of it like defragmenting a hard drive.

```sql
-- Compact all small files in a table
OPTIMIZE table_name;

-- Compact only files matching a predicate (faster)
OPTIMIZE table_name WHERE sale_date >= '2024-10-01';
```

**Important:** `OPTIMIZE` is idempotent — running it twice on the same data does nothing the second time.

---

In [0]:
%sql
-- ============================================================
-- OPTIMIZE: Before and After
-- ============================================================
-- Step 1: Check file count BEFORE optimize
DESCRIBE DETAIL sales_events;

In [0]:
%sql
-- Step 2: Run OPTIMIZE to compact files
OPTIMIZE sales_events;

-- Step 3: Check file count AFTER optimize
-- You should see fewer, larger files
DESCRIBE DETAIL sales_events;

## Part 4: Liquid Clustering — Organizing Data for Your Query Patterns

**Liquid clustering** controls *how data is physically organized in files*. When enabled, Delta Lake groups rows with similar values in the clustering columns into the same files.

**Why this matters:** If your Gold table is clustered by `sale_date` and your dashboard filters by date, the engine can skip 90%+ of files. This is the difference between a 30-second query and a 2-second query.

| Feature | What it does |
| --- | --- |
| `CLUSTER BY (col1, col2)` | Organize data so similar values are in the same files |
| `CLUSTER BY AUTO` | Let Databricks choose clustering keys automatically |
| `ALTER TABLE ... CLUSTER BY` | Change clustering keys at any time (no full rewrite) |

**Key advantages:**
- You can **change clustering columns at any time** (no full rewrite needed)
- It's **self-tuning** and skew-resistant
- Works **incrementally** (only reorganizes new/changed data during OPTIMIZE)
- Databricks can choose keys **automatically** based on usage patterns

**Power BI Parallel:** This is loosely analogous to Power BI aggregations — a way to make queries faster by organizing data for common access patterns. But liquid clustering works at the storage layer, transparently benefiting all queries.

---

In [0]:
%sql
-- ============================================================
-- Liquid Clustering: See how a table is clustered
-- ============================================================
-- Compare the clusteringColumns field between the two tables.

-- Unclustered table: clusteringColumns will be empty []
DESCRIBE DETAIL sales_events;

In [0]:
%sql
-- Clustered table: clusteringColumns shows [sale_date, region]
DESCRIBE DETAIL sales_events_clustered;

In [0]:
%sql
-- ============================================================
-- Change clustering keys on an existing table
-- ============================================================
-- You can change clustering keys at any time without rewriting data!
-- Future OPTIMIZE operations will use the new keys.

-- Enable clustering on a previously unclustered table
ALTER TABLE sales_events
CLUSTER BY (sale_date, region);

-- Verify the change
DESCRIBE DETAIL sales_events;

In [0]:
%sql
-- ============================================================
-- Disable clustering (if needed)
-- ============================================================
-- Setting CLUSTER BY NONE stops future clustering but doesn't
-- rewrite already-clustered data.

ALTER TABLE sales_events
CLUSTER BY NONE;

-- Verify: clusteringColumns should be empty again
DESCRIBE DETAIL sales_events;

## Part 5: Predictive Optimization — The "Set It and Forget It" Solution

For **Unity Catalog managed tables**, Databricks provides **Predictive Optimization** — an automatic maintenance system that eliminates the need to manually run `OPTIMIZE` and `VACUUM`.

**What it does automatically:**
| Operation | Description |
| --- | --- |
| **OPTIMIZE** | Compacts small files and triggers clustering for enabled tables |
| **VACUUM** | Removes old data files no longer referenced by the table |
| **ANALYZE** | Updates statistics for better query planning |

**Is it enabled?** Predictive Optimization is enabled by default for accounts created after November 2024. You can check and configure it at the account, catalog, or schema level.

---

---

## Summary: Delta Lake Optimizations at a Glance

| Concept | What It Does | Your Action |
| --- | --- | --- |
| **DESCRIBE DETAIL** | Shows physical table details (files, size, clustering) | Use to diagnose performance issues |
| **Data Skipping** | Uses min/max stats to skip irrelevant files | Always include WHERE clauses in queries |
| **OPTIMIZE** | Compacts small files into larger ones | Usually automatic via Predictive Optimization |
| **Liquid Clustering** | Organizes data by clustering columns | Ask data engineers to cluster by your filter columns |
| **Predictive Optimization** | Automatic OPTIMIZE + VACUUM + ANALYZE | Enabled by default for managed tables |
